### Converting Given Data To Tensor 
1. In PyTorch, the philosophy is "Explicit is better than implicit." PyTorch generally requires you to manually ensure your data is a torch.Tensor before it enters a model or a specialized data utility.
2. For General Data (Numbers): You usually just use torch.tensor(my_numpy_array).

3. For Images: PyTorch uses transforms.ToTensor(). This does two special things that TensorFlow's basic loading doesn't do automatically:

        1. Permutes Dimensions: It changes an image from (Height, Width, Channels) to (Channels, Height, Width), which is the standard PyTorch format.

        2. Normalizes: It scales pixel values from 0–255 to 0.0–1.0.

In [ ]:
import torch
import numpy as np
arr1 = np.array([1,2,3,4,5]).reshape(5,1)
t1  = torch.tensor(arr1)     # both tensor and Tensor makes tensor but difference is in the dtype!!! Furthermore tensor is the function which can be used for converting numpy arrays whereas Tensor is a Class and it can be used to create tensors directly like torch.Tensor([1,2,3,4])
print(t1.shape)            # tensor() is similar to tf.constant i.e. immutable whereas for parameters which are to be updated during training nn.Parameter can be used 
print(t1)
print("After Flattening")
t1 = t1.flatten()
print(t1.shape)
print(t1)

torch.Size([5, 1])
tensor([[1],
        [2],
        [3],
        [4],
        [5]])
After Flattening
torch.Size([5])
tensor([1, 2, 3, 4, 5])


In [ ]:
# torch.tensor() creates a seperate copy of the numpy array in memory which is use as tensor 
# but torch.from_numpy() creates a tensor which shares the memory with numpy array i.e. point to same underlying buffer 
array = np.array([1,2,3,4,5])
ten = torch.from_numpy(array) # if i add.float() here , now the datatype is different so it will create a seperate memory location for the tensor and thus chnages wont reflect anymore !!!
array[2] = 9
print(ten)

tensor([1, 2, 9, 4, 5])


### DataPipelines

In [1]:

from torch.utils.data import TensorDataset, DataLoader

features_pt = torch.randn(100, 8)
label_pt = torch.randn(100, 1) # already creates tensors

dataset = TensorDataset(features_pt, label_pt)
dataloader = DataLoader(dataset,batch_size = 32, shuffle = True)

print("Pytorch Dataset Ready!!!")

 

Pytorch Dataset Ready!!!


### Model

In [4]:
import torch.nn as nn
import torch.optim as optim

class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(8,16)      # Linear is for torch as how Dense is for tensorflow 
        self.output = nn.Linear(16,1)
        self.relu = nn.ReLU()
    def forward(self, x):
        output = self.relu(self.hidden(x))
        output = self.output(output)
        return output

model_pt = SimpleNet()
criterion = nn.MSELoss()   # for regression 
optimizer = optim.Adam(model_pt.parameters(), lr=0.001)

### Training Mechanics

In [ ]:
model_pt.train()
best_loss = float('inf')
patience = 3
counter = 0
for epoch in range(10):
    for data, target in dataloader:
        optimizer.zero_grad()
        output = model_pt(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
    # for data, target in testloader:       # here as we did not have test and train data seperately so the same train dataset is used for early stopping , but in actual scenerio val_loss is calculated via test loader and then early stopping is checked
        if loss < best_loss:
            best_loss = loss
            counter = 0
            torch.save(model_pt.state_dict(), 'best_model.pt')  # for loading we have load_state_dict() function
        else:
            counter += 1
            if counter >= patience:
                print("Early Stopping Triggered")
                break

In [11]:
print(type(float('inf')))
Y_test = np.random.random((10,1))
Y_test.flatten().shape
type(Y_test)

<class 'float'>


numpy.ndarray

## SimpleAttention Layer – Mathematical Breakdown

### Step 1: Inputs

Suppose your input is a batch of sequences:

$$
X \in \mathbb{R}^{B \times T \times U}
$$

- $B$: batch size  
- $T$: sequence length (number of time steps)  
- $U$: feature dimension (units per step)

---

### Step 2: Learnable weight vector

$$
w \in \mathbb{R}^{U \times 1}
$$

This is a trainable parameter vector.

---

### Step 3: Compute scores

You perform a matrix multiplication:

$$
S = X \cdot w
$$

Shape: $(B \times T \times 1)$.  
This gives a scalar score for each time step in the sequence.

---

### Step 4: Softmax normalization

You normalize across the time dimension:

$$
\alpha_t = \frac{e^{S_t}}{\sum_{j=1}^{T} e^{S_j}}
$$

So the weights $\alpha \in \mathbb{R}^{B \times T \times 1}$ form a probability distribution over time steps.

---

### Step 5: Apply weights

Finally, you scale the input features by these attention weights:

$$
Y_t = X_t \cdot \alpha_t
$$

So the output has the same shape as the input $(B \times T \times U)$, but each time step’s features are reweighted according to its learned importance.

---

### Intuition

This is a very simplified attention mechanism:  
- Instead of computing query–key dot products, it uses a single learned vector $w$ to score each time step.  
- The softmax ensures the scores become a distribution across time steps.  
- Multiplying back by the weights emphasizes “important” time steps and suppresses less relevant ones.
- This is a very simplified attention mechanism: instead of computing query–key dot products, it uses a single learned vector 𝑤 to score each time step.

In [ ]:
class SimpleAttention(nn.Module):
    def __init__(self, input_dim):
        super(SimpleAttention, self).__init__()
        self.w = nn.Paramter(torch.randn(input_dim, 1))

    def forward(self, x):
        scores = torch.matmul(x, self.w)
        weights = torch.softmax(scores, dim = 1)
        return x * weights

# The Identity Layer: A Conceptual Guide

An **Identity Layer** acts as a *pass-through* or a *bridge* within a neural network.  
It takes an input and returns that exact same input as the output, without modifying it or applying any mathematical operations like weights or biases.

---

## 🛠️ Why Do We Use It?

### 1. Residual (Skip) Connections
In deep networks like **ResNet**, identity layers allow the signal to *skip* certain blocks.  
This helps prevent gradients from vanishing during backpropagation by providing a direct path for the gradient to flow through the network.

### 2. Placeholder for Research
When prototyping a new architecture, you can use an identity layer as a temporary placeholder for a complex module or a custom layer that you haven't built or finalized yet.

### 3. Architecture Consistency
Sometimes a specific network architecture requires a layer to exist to keep the code structure or *shape* consistent (for example, in conditional branching), even if that specific branch shouldn't actually change the data.


In [1]:
import torch 
import torch.nn as nn

class IdentityLayer(nn.Module):
    def __init__(self):
        super(IdentityLayer, self).__init__()

    def forward(self, input):
        # return input as it is 
        return input

**Torch Also provides Identity Layer inside 'nn'**

In [ ]:
model = nn.Sequential(
    nn.Linear(8,16),   
    nn.Identity(), # does nothing 
    nn.Linear(16,1)
)
# Dense automaticallu figures out input diemnsion if not specified but Linear in torch expects both inpu and output diemnsion

# Sequence Models

#### In sequence modeling, we usually deal with a 3D Tensor: **(Batch Size, Time Steps, Features).**

#### **PyTorch**: Traditionally, PyTorch defaults to (timesteps, batch, features) because it’s faster for certain optimization libraries (like cuDNN). However, we usually use batch_first=True to make it match the more intuitive (batch, timesteps, features) format.

In [ ]:
import torch
import torch.nn as nn

# nn.LSTM(input_size, hidden_size, num_layers)
lstm_pt = nn.LSTM(5, 64, num_layers=2, batch_first=True)

# Create dummy input: [Batch, Sequence_Length, Features]
input_data = torch.randn(32, 10, 5)

# PyTorch returns: (output, (hidden_state, cell_state))
output, (hn, cn) = lstm_pt(input_data)

# The Output Mystery: Hidden vs. Cell States

An **LSTM** has two types of "memory":
- **Hidden State (\(h\))** → short-term memory  
- **Cell State (\(c\))** → long-term memory  

---

##  Feature Comparison: TensorFlow vs. PyTorch

| Feature            | TensorFlow (`layers.LSTM`)                                                                 | PyTorch (`nn.LSTM`)                                                                 |
|--------------------|---------------------------------------------------------------------------------------------|-------------------------------------------------------------------------------------|
| **Standard Output** | Returns only the hidden state of the last step (or all steps if `return_sequences=True`).   | Returns both the full sequence of hidden states **AND** a tuple containing the final \((h, c)\). |
| **Getting States**  | You must explicitly set `return_state=True` to get the \((h, c)\).                          | It always returns the final states by default.                                      |

---

###  Key Takeaway
- **TensorFlow** is more minimal by default — you only get hidden states unless you ask for cell states.  
- **PyTorch** is more verbose — it always gives you both hidden and cell states, plus the full sequence of hidden states.


# Many to Many and Many to One Architecture

In [ ]:
# Many-to-Many RNN
# batch_first=True makes input (Batch, Seq, Feature)
rnn_pt = nn.RNN(input_size=5, hidden_size=64, batch_first=True)
output, h_final = rnn_pt(input_data) # output contains ALL time steps

# Many-to-One LSTM
lstm_pt = nn.LSTM(5, 64, batch_first=True)
output, (h, c) = lstm_pt(input_data)
# To get 'Many-to-One', we just select the last time step from the output
last_step_output = output[:, -1, :]

output → (batch_size, seq_len, hidden_size)

output[:, -1, :] → (batch_size, hidden_size)

PyTorch always returns (output, (h, c)).

In [ ]:
import torch
import torch.nn as nn

lstm = nn.LSTM(input_size=8, hidden_size=16, batch_first=True)
x = torch.randn(32, 10, 8)  # (batch_size=32, seq_len=10, features=8)

output, (h_n, c_n) = lstm(x)

print(output.shape)   # (32, 10, 16)
print(h_n.shape)      # (1, 32, 16) -> final hidden state
print(c_n.shape)      # (1, 32, 16) -> final cell state

# last time step hidden state
last_output = output[:, -1, :]  
print(last_output.shape)  # (32, 16)


PyTorch nn.LSTM outputs
output[:, -1, :]

Shape: (batch_size, hidden_size)

This is the hidden state at the last time step for each sequence in the batch.

It comes directly from the output tensor, which contains hidden states for all time steps.

h_n

Shape: (num_layers * num_directions, batch_size, hidden_size)

This is the final hidden state for each layer and direction.

For a single-layer, unidirectional LSTM, h_n[0] has shape (batch_size, hidden_size) and matches output[:, -1, :].